# Rede de emissões atribuídas à produção brasileira

## 1. Objetivo

Como se organiza a rede intersetorial de emissões atribuídas à produção brasileira, quais setores ocupam posições mais centrais e de maior alcance, e em que medida os fluxos de emissões se concentram em determinados setores?

Usamos apenas P e o catálogo de setores exportados pela primeira etapa. Não recalculamos a MIP. Os agentes são setores agregados, não empresas. Distinguimos volume, posição estrutural, alcance ponderado e dependência; nenhuma medida isolada demonstra a eficácia de uma intervenção econômica.

In [ ]:
from hashlib import sha256
import json
from importlib.metadata import version
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from redes import dados
from redes.redes import carregar_matriz_emissoes, carregar_setores, matriz_para_grafo
from redes.visualizacoes import figura_mapa_calor, figura_setor, figura_rede, exportar_pagina_redes
pd.set_option("display.max_columns", 16)
pd.set_option("display.max_rows", 70)

## 2. Inputs e definição da rede

`outputs/matriz_emissoes_producao_2015.csv` contém P, em Gg de CO₂; `outputs/setores_2015.csv` identifica os setores. O manifesto existente verifica seus hashes antes da leitura. São outputs regeneráveis: execute a exportação da etapa MIP se não existirem. Hashes não são atualizados automaticamente.

**i → j** significa emissões da atividade i atribuídas à demanda final pelo bem da atividade j, incluindo exportações. A matriz já contém efeitos diretos e indiretos; não representa transações diretas ou caminhos físicos do carbono.

Separamos P da rede intersetorial W, obtida zerando apenas a diagonal. A diagonal permanece registrada como atribuição intrassetorial. Mantemos todos os nós, inclusive isolados, e todas as arestas positivas, sem threshold nos cálculos. Não carregamos C, produção bruta ou intensidades adicionais.

In [ ]:
P = carregar_matriz_emissoes("matriz_emissoes_producao_2015")
setores = carregar_setores("setores_mip_2015")
assert P.index.equals(setores.index) and P.shape == (67, 67)
entradas = pd.DataFrame([dados.entrada(i) for i in ["matriz_emissoes_producao_2015", "setores_mip_2015"]])
display(entradas[["id", "arquivo", "sha256"]])
print(f"P: {P.shape[0]} emissores × {P.shape[1]} destinos, em Gg de CO₂")
display(setores.to_frame())

In [ ]:
# P é preservada. W representa somente atribuições entre atividades distintas.
diagonal = pd.Series(np.diag(P), index=P.index, name="diagonal")
valores = P.to_numpy(copy=True)
np.fill_diagonal(valores, 0)
W = pd.DataFrame(valores, index=P.index, columns=P.columns)
G = matriz_para_grafo(W)
nx.set_node_attributes(G, setores.to_dict(), "descricao")
np.testing.assert_allclose(W.to_numpy().sum() + diagonal.sum(), P.to_numpy().sum())
assert list(G) == P.index.tolist() and nx.number_of_selfloops(G) == 0

## 3. Estrutura geral e volumes

Graus contam relações; forças somam pesos. A força de saída identifica atribuições de um emissor a outros destinos; a força de entrada identifica emissões de outros setores atribuídas ao bem demandado. Ambas excluem a diagonal.

`emissoes_totais_i = Σ_j P_ij` inclui a diagonal. Não confundir um setor isolado em W com um setor sem emissões. A densidade é `m/[n(n−1)]`. Componentes fracas ignoram a direção; fortes exigem caminhos nos dois sentidos. Usamos essas medidas apenas para descrever a estrutura, pois efeitos indiretos tornam a rede muito densa.

In [ ]:
metricas = pd.DataFrame({"descricao": setores})
metricas["emissoes_totais"] = P.sum(axis=1)
metricas["diagonal"] = diagonal
metricas["forca_saida"] = pd.Series(dict(G.out_degree(weight="weight")))
metricas["forca_entrada"] = pd.Series(dict(G.in_degree(weight="weight")))
metricas["grau_saida"] = pd.Series(dict(G.out_degree()))
metricas["grau_entrada"] = pd.Series(dict(G.in_degree()))
metricas["participacao_total"] = metricas["emissoes_totais"] / metricas["emissoes_totais"].sum() if P.to_numpy().sum() else 0
metricas["participacao_intersetorial"] = metricas["forca_saida"] / metricas["forca_saida"].sum() if G.size(weight="weight") else 0
np.testing.assert_allclose(metricas["forca_saida"] + diagonal, P.sum(axis=1))
np.testing.assert_allclose(metricas["forca_entrada"] + diagonal, P.sum(axis=0))
resumo = pd.DataFrame([{
    "nos": len(G), "arestas": G.number_of_edges(), "densidade": nx.density(G),
    "peso_total_gg": P.to_numpy().sum(), "peso_intersetorial_gg": G.size(weight="weight"),
    "diagonal_gg": diagonal.sum(), "isolados": nx.number_of_isolates(G),
    "componentes_fracas": nx.number_weakly_connected_components(G),
    "componentes_fortes": nx.number_strongly_connected_components(G),
}], index=pd.Index(["producao"], name="rede"))
display(resumo)
display(metricas.sort_values("emissoes_totais", ascending=False).head(10))

## 4. Concentração entre emissores

Calculamos separadamente dois universos: emissões totais por emissor, incluindo diagonal, e atribuições intersetoriais por emissor, sem diagonal. Em ambos entram todos os setores, inclusive valores zero.

Se `s_i` é a participação do emissor, **HHI = Σ_i s_i²**: vale 1/n na distribuição igual e 1 na concentração em um emissor. O **Gini** compara diferenças entre todos os pares: `Σ_i Σ_j |e_i−e_j| / (2n Σ_i e_i)`. Sem correção amostral, seu máximo com n setores é `(n−1)/n`.

A curva de Lorenz ordena do menor para o maior emissor; a curva de participação acumulada ordena do maior para o menor. Top 5 e top 10 mostram a parcela emitida pelos maiores em cada universo. Por convenção computacional, se o peso total for zero, índices e participações valem zero; nesse caso não há distribuição positiva para interpretar.

In [ ]:
universos = {"total_com_diagonal": metricas["emissoes_totais"],
             "intersetorial_sem_diagonal": metricas["forca_saida"]}
concentracao = {}
curvas = []
for nome, serie in universos.items():
    valores = serie.to_numpy(dtype=float)
    n = len(valores)
    total = valores.sum()
    participacoes = valores / total if total else np.zeros(n)
    crescente = np.sort(participacoes)
    concentracao[nome] = {
        "peso_gg": total,
        "top5": np.sort(participacoes)[::-1][:5].sum(),
        "top10": np.sort(participacoes)[::-1][:10].sum(),
        "hhi": np.square(participacoes).sum(),
        "gini": np.abs(valores[:, None] - valores[None, :]).sum() / (2 * n * total) if total else 0,
    }
    curvas.append(pd.DataFrame({"universo": nome, "fracao_setores": np.arange(n + 1) / n,
        "lorenz": np.r_[0, np.cumsum(crescente)],
        "acumulado_maiores": np.r_[0, np.cumsum(crescente[::-1])]}))
concentracao = pd.DataFrame.from_dict(concentracao, orient="index").rename_axis("universo")
curvas = pd.concat(curvas, ignore_index=True)
display(concentracao)

## 5. Centralidade estrutural além do volume

PageRank ponderado segue ligações proporcionalmente ao peso, com amortecimento 0,85 e teletransporte uniforme. Nós sem saída distribuem probabilidade uniformemente. O escore soma 1.

No grafo original, destaca **destinos** que recebem atribuições de origens importantes. No grafo invertido, destaca **emissores** relacionados a destinos importantes. Não são quantidades de emissões. A diagonal permanece excluída.

Comparamos PageRank emissor com força de saída, e PageRank destino com força de entrada. Posto 1 é o maior valor; empates recebem posto médio. Uma diferença positiva `posto_forca − posto_pagerank` indica promoção pelo PageRank. Correlação dos postos mede concordância; se um ranking for constante, a correlação fica indefinida. Forte concordância limita o ganho de informação em relação ao volume.

[Definição e parâmetros do NetworkX](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html). Não calculamos caminhos mínimos ou um índice agregado de criticidade.

In [ ]:
metricas["pagerank_destino"] = pd.Series(nx.pagerank(G, alpha=.85, weight="weight", max_iter=1000, tol=1e-12))
metricas["pagerank_emissor"] = pd.Series(nx.pagerank(G.reverse(copy=False), alpha=.85, weight="weight", max_iter=1000, tol=1e-12))
np.testing.assert_allclose(metricas[["pagerank_destino", "pagerank_emissor"]].sum(), [1, 1])
rankings = metricas[["emissoes_totais", "forca_saida", "forca_entrada", "pagerank_emissor", "pagerank_destino"]].rank(ascending=False, method="average")
rankings["promocao_emissor"] = rankings["forca_saida"] - rankings["pagerank_emissor"]
rankings["promocao_destino"] = rankings["forca_entrada"] - rankings["pagerank_destino"]
rankings.insert(0, "descricao", setores)
for papel, forca in [("emissor", "forca_saida"), ("destino", "forca_entrada")]:
    a, b = rankings[forca], rankings["pagerank_" + papel]
    resumo["correlacao_rank_" + papel] = a.corr(b) if a.nunique() > 1 and b.nunique() > 1 else np.nan
display(rankings.sort_values("pagerank_emissor").head(10))
display(rankings.reindex(rankings["promocao_emissor"].abs().sort_values(ascending=False).index).head(10))
display(resumo[["correlacao_rank_emissor", "correlacao_rank_destino"]])

## 6. Alcance ponderado e dependência

### 6.1. Distribuição das saídas

`q_ij = W_ij / Σ_j W_ij` descreve como cada emissor distribui suas atribuições. O **número efetivo de destinos = 1 / Σ_j q_ij²** vale 1 para concentração em um destino e k para k destinos igualmente relevantes. Um emissor sem saídas recebe zero. Esse alcance ponderado distingue setores com o mesmo grau e volume, mas distribuições diferentes; não mede alcançabilidade por caminhos.

### 6.2. Dependência dos destinos

`d_ij = W_ij / Σ_i W_ij` mede a participação de i nas atribuições intersetoriais recebidas por j. Cada coluna não nula soma 1; colunas sem entradas ficam zeradas. Isso não é a parcela de insumos físicos comprada de i. Um destino pode apresentar alta dependência relativa com pequeno volume absoluto, por isso mostramos participação e peso juntos.

Para cada destino, listamos os três principais emissores; para cada emissor, os três destinos com maior dependência relativa. A seleção é apenas uma síntese das tabelas: a matriz completa de dependência é exportada. Empates nos recortes são ordenados pelo código setorial.

In [ ]:
q = W.div(metricas["forca_saida"].replace(0, np.nan), axis=0).fillna(0)
hhi_destinos = q.pow(2).sum(axis=1)
metricas["destinos_efetivos"] = 1 / hhi_destinos.replace(0, np.nan)
metricas["destinos_efetivos"] = metricas["destinos_efetivos"].fillna(0)
d = W.div(metricas["forca_entrada"].replace(0, np.nan), axis=1).fillna(0)
np.testing.assert_allclose(d.sum(axis=0), (metricas["forca_entrada"] > 0).astype(float))
np.testing.assert_allclose(q.sum(axis=1), (metricas["forca_saida"] > 0).astype(float))
metricas["maior_dependencia_destino"] = d.max(axis=1)
rankings["destinos_efetivos"] = metricas["destinos_efetivos"].rank(ascending=False, method="average")
principais_emissores = []
destinos_dependentes = []
for destino in d.columns:
    for emissor, participacao in d[destino].sort_index().sort_values(ascending=False, kind="stable").head(3).items():
        if participacao > 0:
            principais_emissores.append({"destino": destino, "descricao_destino": setores[destino],
                "emissor": emissor, "descricao_emissor": setores[emissor], "participacao": participacao, "peso_gg": W.loc[emissor, destino]})
for emissor in d.index:
    for destino, participacao in d.loc[emissor].sort_index().sort_values(ascending=False, kind="stable").head(3).items():
        if participacao > 0:
            destinos_dependentes.append({"emissor": emissor, "descricao_emissor": setores[emissor],
                "destino": destino, "descricao_destino": setores[destino], "participacao": participacao, "peso_gg": W.loc[emissor, destino]})
principais_emissores = pd.DataFrame(principais_emissores)
destinos_dependentes = pd.DataFrame(destinos_dependentes)
display(metricas[["descricao", "emissoes_totais", "forca_saida", "destinos_efetivos"]].sort_values("destinos_efetivos", ascending=False).head(10))
display(principais_emissores)
display(destinos_dependentes)

## 7. Visualizações e exploração

Lorenz e participação acumulada distinguem os dois universos de concentração. A dispersão relaciona volume total e alcance efetivo, mantendo os setores sem saídas. Os mapas de W e d seguem a mesma ordem setorial: magnitude em `log10(1 + Gg)` e dependência de 0 a 100%, com diagonal mascarada e valores originais no hover.

Na exploração por setor mostramos até dez entradas e saídas, calculadas sobre toda W, com cobertura e diagonal. O desenho não pressupõe equilíbrio das entradas e saídas nem conservação de fluxos pelo setor central.

A visão geral secundária mostra as 75 maiores arestas e todos os nós. Tamanho = `8 + 27 sqrt(emissoes_totais/maximo)`; cor = destinos efetivos; espessura = `0,5 + 4 sqrt(peso/maximo)`. O tamanho inclui diagonal; as arestas não. O círculo segue a ordem dos inputs. O layout por forças usa a rede visual sem pesos de atração e semente 42. Nenhum filtro ou layout altera as métricas.

In [ ]:
figuras = {}
figuras["Curva de Lorenz"] = px.line(curvas, x="fracao_setores", y="lorenz", color="universo", template="plotly_white",
    labels={"fracao_setores": "Fração dos setores (menor → maior)", "lorenz": "Fração das emissões"})
figuras["Curva de Lorenz"].add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(color="gray", dash="dot"))
figuras["Participação acumulada dos maiores"] = px.line(curvas, x="fracao_setores", y="acumulado_maiores", color="universo", template="plotly_white",
    labels={"fracao_setores": "Fração dos setores (maior → menor)", "acumulado_maiores": "Fração das emissões"})
for figura in figuras.values():
    figura.update_layout(legend=dict(orientation="h", y=-.35), margin=dict(b=125), height=520)
figuras["Volume e alcance dos emissores"] = px.scatter(metricas.reset_index(), x="emissoes_totais", y="destinos_efetivos",
    color="maior_dependencia_destino", hover_name="descricao", hover_data=["atividade", "forca_saida", "pagerank_emissor", "diagonal"],
    labels={"emissoes_totais": "Emissões totais (Gg, incluindo diagonal)", "destinos_efetivos": "Número efetivo de destinos",
            "maior_dependencia_destino": "Maior dependência"}, template="plotly_white")
max_peso = float(W.to_numpy().max())
figuras["Magnitude intersetorial"] = figura_mapa_calor(W, setores, "Emissões atribuídas (Gg)", max_peso)
figuras["Dependência dos destinos"] = figura_mapa_calor(d, setores, "Participação no destino", 1, dependencia=True)

estados = []
for setor in setores.index:
    entradas_setor = sorted(G.in_edges(setor, data=True), key=lambda e: (-e[2]["weight"], e[0]))[:10]
    saidas_setor = sorted(G.out_edges(setor, data=True), key=lambda e: (-e[2]["weight"], e[1]))[:10]
    total_entrada, total_saida = metricas.loc[setor, ["forca_entrada", "forca_saida"]]
    estados.append({"setor": setor, "descricao": setores[setor],
        "entradas": [(i, setores[i], a["weight"]) for i, _, a in entradas_setor],
        "saidas": [(j, setores[j], a["weight"]) for _, j, a in saidas_setor],
        "cobertura_entrada": 100 * sum(a["weight"] for _, _, a in entradas_setor) / total_entrada if total_entrada else 0,
        "cobertura_saida": 100 * sum(a["weight"] for _, _, a in saidas_setor) / total_saida if total_saida else 0,
        "diagonal": diagonal[setor]})
figuras["Explorar um setor"] = figura_setor(estados, "Entradas → setor selecionado → saídas")
arestas_visuais = sorted(G.edges(data=True), key=lambda e: (-e[2]["weight"], e[0], e[1]))[:75]
cobertura = pd.DataFrame([{"arestas_exibidas": len(arestas_visuais), "arestas_calculo": G.number_of_edges(),
    "peso_exibido_pct": 100 * sum(a["weight"] for _, _, a in arestas_visuais) / G.size(weight="weight") if G.size(weight="weight") else 0}], index=["producao"])
posicoes = nx.circular_layout(list(setores.index))
circular = figura_rede(arestas_visuais, metricas, posicoes, "Tamanho: emissões · cor: alcance", max_peso, metricas["emissoes_totais"].max())
layout_visual = nx.Graph()
layout_visual.add_nodes_from(setores.index)
layout_visual.add_edges_from((i, j) for i, j, _ in arestas_visuais)
posicoes_forcas = nx.spring_layout(layout_visual, seed=42, weight=None, k=.8, iterations=200)
alternativa = figura_rede(arestas_visuais, metricas, posicoes_forcas, "Tamanho: emissões · cor: alcance", max_peso, metricas["emissoes_totais"].max())
# Os dois layouts compartilham arestas, atributos e escalas.
figuras = {"Circular": circular, "Por forças": alternativa, **figuras}
display(cobertura)
for figura in figuras.values(): display(figura)

## 8. Síntese: volume, alcance e dependência

As frases abaixo são regeneradas a partir dos resultados. Criticidade é discutida como relevância em dimensões distintas; não há escore composto. Um emissor grande pode ter alcance restrito, e um emissor menor pode dominar atribuições de determinados destinos. Dependência é relativa às atribuições intersetoriais, não às emissões totais do destino.

In [ ]:
conclusoes = []
for universo, linha in concentracao.iterrows():
    conclusoes.append(f"No universo {universo}, os cinco maiores concentram {linha.top5:.1%} e os dez maiores {linha.top10:.1%}; Gini = {linha.gini:.3f}, HHI = {linha.hhi:.3f}.")
maior = metricas["emissoes_totais"].idxmax()
amplo = metricas["destinos_efetivos"].idxmax()
conclusoes.append(f"{maior} — {setores[maior]} lidera as emissões totais ({metricas.loc[maior, 'participacao_total']:.1%}). "
                 f"{amplo} — {setores[amplo]} apresenta o maior alcance ponderado ({metricas.loc[amplo, 'destinos_efetivos']:.1f} destinos efetivos).")
for papel in ["emissor", "destino"]:
    corr = resumo.iloc[0]["correlacao_rank_" + papel]
    conclusoes.append(f"A correlação dos rankings de PageRank e força para {papel} é {corr:.3f}. "
                     "Quanto mais próxima de 1, menor a mudança de ordenação em relação ao volume.")
mudanca = rankings["promocao_emissor"].abs().idxmax()
conclusoes.append(f"Maior divergência entre força de saída e PageRank emissor: {mudanca} — {setores[mudanca]}, "
                 f"postos {rankings.loc[mudanca, 'forca_saida']:g} e {rankings.loc[mudanca, 'pagerank_emissor']:g}, respectivamente.")
if not principais_emissores.empty:
    relacao = principais_emissores.sort_values("participacao", ascending=False).iloc[0]
    conclusoes.append(f"A dependência relativa mais elevada ocorre em {relacao.destino} — {relacao.descricao_destino}: "
                     f"{relacao.participacao:.1%} das atribuições intersetoriais vêm de {relacao.emissor} — {relacao.descricao_emissor} "
                     f"({relacao.peso_gg:,.2f} Gg).")
conclusoes.append("Essas posições identificam relações relevantes para investigação; não demonstram a redução de emissões que resultaria de intervir em um setor.")
for texto in conclusoes: print(texto + "\n")

## 9. Exportação e página interativa

Exporta somente os resultados da rede de produção para `outputs/redes/`: métricas, rankings, concentração, matriz de dependência, resumo, cobertura visual e proveniência. As participações dos CSVs usam frações de 0 a 1. Os recortes de três relações por setor são mostrados no notebook e na página; a matriz completa permite reconstruí-los.

`docs/index.html` incorpora figuras, tabelas e downloads, sem backend ou recálculo no navegador. A página registra os hashes dos inputs e das fontes das células (sem outputs/contadores), os módulos e versões. O notebook sobrescreve os outputs atuais; publicar por GitHub Pages continua sendo uma etapa separada. Nenhum arquivo da MIP é modificado.

In [ ]:
pasta_outputs = dados.RAIZ_PROJETO / "outputs" / "redes"
pasta_outputs.mkdir(parents=True, exist_ok=True)
tabelas_saida = {"metricas_producao": metricas, "rankings_producao": rankings,
    "concentracao_producao": concentracao, "dependencia_producao": d,
    "resumo_redes": resumo, "cobertura_visual": cobertura}
for nome, tabela in tabelas_saida.items():
    tabela.to_csv(pasta_outputs / (nome + ".csv"), encoding="utf-8", lineterminator="\n")
# Recortes para consulta; a exportação canônica é a matriz completa de dependência.
tabelas_saida["principais_emissores_por_destino"] = principais_emissores
tabelas_saida["destinos_dependentes_por_emissor"] = destinos_dependentes
registros = [{"tipo": "input", "arquivo_ou_pacote": r["arquivo"], "sha256_ou_versao": r["sha256"]}
             for _, r in entradas.iterrows()]
fonte_notebook = json.loads((dados.RAIZ_PROJETO / "analise_redes_emissoes.ipynb").read_text(encoding="utf-8"))
fontes = json.dumps([c["source"] for c in fonte_notebook["cells"]], ensure_ascii=False).encode("utf-8")
registros.append({"tipo": "fontes das celulas", "arquivo_ou_pacote": "analise_redes_emissoes.ipynb", "sha256_ou_versao": sha256(fontes).hexdigest()})
for arquivo in ["redes/redes.py", "redes/dados.py", "redes/visualizacoes.py", "requirements.txt"]:
    registros.append({"tipo": "codigo", "arquivo_ou_pacote": arquivo,
                      "sha256_ou_versao": sha256((dados.RAIZ_PROJETO / arquivo).read_bytes()).hexdigest()})
for pacote in ["networkx", "pandas", "numpy", "scipy", "plotly"]:
    registros.append({"tipo": "versao", "arquivo_ou_pacote": pacote, "sha256_ou_versao": version(pacote)})
proveniencia = pd.DataFrame(registros)
proveniencia.to_csv(pasta_outputs / "proveniencia.csv", index=False, encoding="utf-8", lineterminator="\n")
exportar_pagina_redes(dados.RAIZ_PROJETO / "docs" / "index.html", figuras, tabelas_saida, conclusoes, proveniencia, cobertura)
print("CSVs: outputs/redes/ | Página interativa: docs/index.html")